In [ ]:
import sympy as sp
import numpy as np
from scipy.special import jv
from scipy.sparse import lil_matrix
import matplotlib.pyplot as plt
from scipy.linalg import block_diag
from scipy.optimize import root
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import Plot_setting
from Plot_setting import *

# Hill method — period-2 orbit (corrected 08/10/2026)

Copy of `Hill_method_doubling.ipynb` with these changes:

1. **Floquet exponents**: `Floq_coef` keeps the **2n = 4** genuine exponents (Detroux et al. 2015), not 4n.
   They are selected by eigenvector centring, keeping one member per family $\lambda + i m$. Plots use all 2n exponents.
   Check: $\sum \mathrm{Re}\,\lambda = -\mathrm{tr}(M^{-1}C)$ (Liouville), printed after each sweep.
2. **Damping**: same convention as the Julia simulations (`SystemATS`: $\ddot\phi_b = \dots - 2\kappa_b\dot\phi_b$).
   With $\tau_d = \tau/2$ this gives $C = \mathrm{diag}(0, \kappa_b'/2)$. That is the value the code already used;
   only the markdown is updated ($2\kappa_b p_b$). It is now consistent with `Hill_method_corrected.ipynb`
   ($C = \kappa_b'$ in $\tau$) and with $\kappa_b = \pi/10.4$.
3. **2D sweep**: `Floq_coef` returns `(eigvals, eigvecs)`, so only the eigenvalues are stored. They are stored along the
   **ascending** `eps_p_list`. The old cell mixed the reversed loop index with the ascending list, which mirrored the
   threshold inside [0.6, 0.72]. The output JSON is ordered along the ascending `eps_d_ratio_list`, so plot it against
   `alpha_sq_Hill` **without** `[::-1]`.
4. **2D sweep seeding**: the old seed `z_guess_0`, used directly at $\varepsilon_p = 0.72$, converges to the stable period-1
   (cat) orbit, so no instability was found. The sweep now starts from the period-2 orbit of the 1D sweep
   ($\varepsilon_p = 1.2$, $\varepsilon_d/g_2 = 10.12$). It continues down to $\varepsilon_p = 0.72$, then across $\varepsilon_d/g_2$,
   then down in $\varepsilon_p$ for each ratio. `odd_content_array` records the period-2 content of mode a
   (it vanishes when the orbit merges back with the period-1 orbit at the period doubling).

`db_dz` is vectorised; its result is unchanged. Outputs are saved under new names (suffix `_08_10_2026`).

**12/10/2026: definition of $g_2$.** The pump is static ($\omega_p = 0$) and enters the EOMs as $\sin\varepsilon_p$, so
$g_2 = E_J\sin(\varepsilon_p)\,\varphi_a^2\varphi_b$ (`g2_of`), as in the article. It was $J_1(\varepsilon_p)E_J\varphi_a^2\varphi_b$
(oscillating pump, Leon's paper), about half as large. The drive is still $\varepsilon_d = (\varepsilon_d/g_2)\,g_2$, so at a given ratio
$\varepsilon_d$ is now about twice as large and $\varepsilon_d/2g_2 = |\alpha|^2$ at lowest order. The hard-coded seed (period-2 orbit at
$\varepsilon_p = 1.2$) was converged with the former definition (`g2_J1_of`): each sweep first continues it in $\varepsilon_d$ up to the new
value. The windows of the sweeps are adapted to the new thresholds. Outputs have the suffix `_12_10_2026`.


Hamiltonian $$H = \omega_a(x_a^2/2+p_a^2/2) + \omega_b(x_b^2/2+p_b^2/2) + \varepsilon_d\cos(\omega_d t)\sqrt 2 x_b-2E_J\sin(\varepsilon_p)[\sin(\sqrt 2 \varphi_ax_a+ \sqrt 2 \varphi_bx_b)-\sqrt 2 \varphi_ax_a- \sqrt 2 \varphi_bx_b]$$

EOMs
\begin{equation}
\begin{cases}
\dot x_a = \omega_a p_a \\
\dot p_a = -\omega_a x_a +2E_J\sin(\varepsilon_p)\sqrt 2 \varphi_a[\cos(\sqrt 2 \varphi_ax_a+ \sqrt 2 \varphi_bx_b)-1] \\
\dot x_b = \omega_b p_b \\
\dot p_b = -\omega_b x_b - \sqrt 2 \varepsilon_d\cos(\omega_d t)+2E_J\sin(\varepsilon_p)\sqrt 2 \varphi_b[\cos(\sqrt 2 \varphi_ax_a+ \sqrt 2 \varphi_bx_b)-1] -2\kappa_b p_b\\
\end{cases}
\end{equation}

We rescale time and the variables
\begin{equation}
\begin{cases}
 dx_a^{\lambda}/d(2\tau) = p_a^{\lambda} \\
dp_a^{\lambda}/d(2\tau) = - x_a^{\lambda} +2E_J\sin(\varepsilon_p)\sqrt 2 \varphi_a^2 [\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1]/\omega_a \\
dx_b^{\lambda}/d(4\tau) = p_b^{\lambda} \\
dp_b^{\lambda}/d(4\tau) = -x_b^{\lambda} - \sqrt 2 \varepsilon_d\varphi_b\cos(4\tau)/\omega_b+2E_J\sin(\varepsilon_p)\sqrt 2 \varphi_b^2 [\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1]/\omega_b-2\kappa_b p_b^{\lambda}/\omega_b\\
\end{cases}
\end{equation}

\begin{equation}
\begin{cases}
 dx_a^{\lambda}/d(2\tau) = p_a^{\lambda} \\
dp_a^{\lambda}/d(2\tau) = - x_a^{\lambda} +\sqrt 2\beta_a' [\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1] \\
dx_b^{\lambda}/d(4\tau) = p_b^{\lambda} \\
dp_b^{\lambda}/d(4\tau) = -x_b^{\lambda} - \sqrt 2 \varepsilon_d'\cos(4\tau)+\sqrt 2\beta_b'[\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1]-2\kappa_b' p_b^{\lambda}\\
\end{cases}
\end{equation}

Finally we write the second order equations on the positions

\begin{equation}
\begin{cases}
\frac{1}{4}\frac{d^2x_a^{\lambda}}{d\tau^2} = - x_a^{\lambda} +\sqrt 2\beta_a' [\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1] \\
\frac{1}{16}\frac{d^2x_b^{\lambda}}{d\tau^2} = -x_b^{\lambda} - \sqrt 2 \varepsilon_d'\cos(4\tau)+\sqrt 2\beta_b'[\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1] -\frac{\kappa_b'}{2} dx_b^{\lambda}/d\tau\\
\end{cases}
\end{equation}

We introduce $\bold X = (x_a^{\lambda}, x_b^{\lambda})^T$ and we get

\begin{equation}
\begin{split}
\begin{pmatrix}1/4&0\\0&1/16\end{pmatrix}\ddot{\bold X}+\begin{pmatrix}0&0\\0&\kappa_b'/2\end{pmatrix}\dot{\bold X}+\begin{pmatrix}1&0\\0&1\end{pmatrix}\bold X-\begin{pmatrix}\beta_a'\\\beta_b'\end{pmatrix}\sqrt 2[\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1] =& -\begin{pmatrix}0\\1\end{pmatrix}\sqrt 2 \varepsilon_d'\cos(4\tau)\\
M\ddot{\bold X} + C\dot{\bold X} + K\bold X + \bold f_{nl}(\bold X)= \bold f_{ext}(\omega = 4,\tau)
\end{split}
\end{equation}

Then we are going to write the position as fFourier series
$$\bold X(t)=\frac{\bold c_0^x}{\sqrt 2}+\sum_{k=1}^{N_H}\biggl(\bold s_k^x\sin(kt)+\bold c_k^x\cos(kt)\biggr)$$

Let $$\bold f_{nl}(\bold X) = \frac{\bold c_0^f}{\sqrt 2}+\sum_{k=1}^{N_H}\biggl(\bold s_k^f\sin(kt)+\bold c_k^f\cos(kt)\biggr)$$

Let $z = [\bold c_0^x, \bold s_1^x, \bold c_1^x, ...]$ and $b = [\bold c_0^f, \bold s_1^f, \bold c_1^f, ...]$

In [ ]:
h_bar=1
e=1
phi_quantum=1/2

#### Caracterization of the circuit #####

w_a = 25.338776456203686
w_b = 2*w_a

phi_a = 0.11
Z_a = 2*phi_quantum**2*phi_a**2
C_a_inv = Z_a*w_a
E_Ca = C_a_inv/2

phi_b = 0.204
Z_b = 2*phi_quantum**2*phi_b**2
C_b_inv = Z_b*w_b
E_Cb = C_b_inv/2

E_J = 37.12*2*np.pi*1.5

#Environment
kappa_b=np.pi/10.4; # 1/ns, same value and convention as the Julia notebooks (phi_b'' = ... - 2 kappa_b phi_b')

w_d=w_b # Drive frequency
w_p=2*w_a-w_d # Pump Frequency

epsilon_p = 1.2
def g2_of(eps_p):
    """Two-photon rate for the static pump (w_p = 0), as in the EOMs: g2 = E_J sin(eps_p) phi_a^2 phi_b."""
    return np.sin(eps_p)*E_J*phi_a**2*phi_b

def g2_J1_of(eps_p):
    """Former definition (oscillating pump, Leon's paper). Only used to continue the seeds below, which were converged with it."""
    return jv(1, eps_p)*E_J*phi_a**2*phi_b

g2 = g2_of(epsilon_p)
epsilon_d = 10.12*g2#0.75*g2_max

beta_a_p, beta_b_p = 2*E_J*np.sin(epsilon_p)*phi_a**2/(w_a), 2*E_J*np.sin(epsilon_p)*phi_b**2/(w_b)
B = np.array([beta_a_p, beta_b_p])  # shape (2,)
eps_d_p = epsilon_d*phi_b/w_b
kappa_b_p = kappa_b / w_b

z_0_init = np.array([-8.67480681e-02, -1.49178000e-01, -1.55487349e-04, -3.92476650e-06,
         4.80123540e-02,  6.60523141e-02, -3.00220194e-02,  3.07073460e-16,
         3.31363194e-01, -1.72222278e-16, -3.88737312e-03,  1.57262610e-02,
         3.35330370e-02, -1.65080648e-01,  6.93437379e-04, -5.59462961e-02,
        -4.67601081e-03,  3.00083819e-01,  4.52017869e-05,  9.32575051e-04,
        -4.88538089e-04, -7.81645147e-03, -1.26778042e-03, -1.46772441e-02,
         4.61868963e-03,  5.06227939e-02,  2.68514538e-04,  2.61445320e-03,
        -1.01232565e-03, -9.46918403e-03, -5.89963411e-04, -5.17649744e-03,
         1.52354566e-03,  1.30588333e-02])

In [ ]:
def build_L(M, C, K, N_H, omega=1.0, nu=1.0):
    """
    Build linear HB operator L for ordering:
    [c0, s1, c1, s2, c2, ..., s_NH, c_NH] with each block in R^n.
    Returns L of shape (n*(2N_H+1), n*(2N_H+1)).
    """
    n = M.shape[0]
    dim = n * (2*N_H + 1)
    L = np.zeros((dim, dim), dtype=float)

    # helper to locate block indices in z
    def idx_c0():      return 0
    def idx_sk(k):     return 1 + 2*(k-1)
    def idx_ck(k):     return 2 + 2*(k-1)

    # DC: (K) acting on c0 (with your 1/sqrt2 convention, L is still K here
    # because the same scaling appears in Gamma and Gamma^+; keep L=K)
    i0 = idx_c0()
    L[i0*n:(i0+1)*n, i0*n:(i0+1)*n] = K

    # Harmonics
    for k in range(1, N_H+1):
        A = K - ((k*omega/nu)**2)*M  # n×n
        B = (k*omega/nu) * C          # n×n

        is_ = idx_sk(k)
        ic_ = idx_ck(k)

        # sine row blocks
        L[is_*n:(is_+1)*n, is_*n:(is_+1)*n] = A
        L[is_*n:(is_+1)*n, ic_*n:(ic_+1)*n] = -B

        # cosine row blocks
        L[ic_*n:(ic_+1)*n, is_*n:(is_+1)*n] = B
        L[ic_*n:(ic_+1)*n, ic_*n:(ic_+1)*n] = A

    return L

def build_b_ext(N_H, eps_d_p, nu = 2):
    """
    b_ext in the same ordering as z:
    [c0, s1, c1, s2, c2, ..., s_NH, c_NH], each block in R^2.
    Forcing: f_ext = -sqrt(2)*eps_d' * cos(2 tau) * e2.
    """
    n = 2
    b = np.zeros(n*(2*N_H+1), dtype=float)

    def idx_ck(k): return 2 + 2*(k-1)

    if N_H >= nu:
        ic2 = idx_ck(nu)
        # e2 = [0,1]
        b[ic2*n:(ic2+1)*n] = np.array([0.0, -np.sqrt(2.0)*eps_d_p])

    return b

In [ ]:
def build_Gamma(t, N_H, n, omega=2.0, nu=2.0):
    """
    Build Gamma = I_n ⊗ Phi with omega/nu = 1.

    t   : array of shape (N,)
    N_H : number of harmonics
    n   : number of DOFs
    """
    t = np.asarray(t)
    N = t.size
    k = np.arange(1, N_H + 1)

    # N x N_H
    S = np.sin(np.outer(t, k*omega/nu))
    C = np.cos(np.outer(t, k*omega/nu))

    # Build Phi: N x (2*N_H + 1) with columns [1/sqrt2, sin(1t), cos(1t), sin(2t), cos(2t), ...]
    Phi = np.empty((N, 2 * N_H + 1), dtype=float)
    Phi[:, 0] = 1.0 / np.sqrt(2.0)
    Phi[:, 1::2] = S
    Phi[:, 2::2] = C

    # Gamma: (n*N) x (n*(2*N_H+1))
    Gamma = np.kron(Phi, np.eye(n))
    return Gamma

def x_tilde_to_X(tildex, n, N):
    # tildex is time-major: [x(t1); x(t2); ...], each x(tj) is length n
    return tildex.reshape(N, n).T   # -> (n, N)

def X_to_x_tilde(X):
    # X is (n, N); return time-major vector
    return X.T.reshape(-1)

def f_nl_time(X):
    """
    X: shape (n, N) with n=2
    returns Fnl: shape (n, N)
    """
    y = np.sqrt(2.0) * (X[0, :] + X[1, :])          # shape (N,)
    g = np.cos(y) - 1.0                             # shape (N,)
    Fnl = -np.sqrt(2.0) * B[:, None] * g[None, :]    # (2,N)
    return Fnl

def b_nl(z, Gamma, Gamma_pinv, n, N):
    x_tilde = Gamma @ z
    X = x_tilde_to_X(x_tilde, n, N)
    Fnl = f_nl_time(X)              # (n,N)
    f_tilde = X_to_x_tilde(Fnl)     # (nN,)
    return Gamma_pinv @ f_tilde     # (n*(2N_H+1),)

In [ ]:
def residual(z, L, b_ext, Gamma, Gamma_pinv, n, N):
    return L @ z + b_nl(z, Gamma, Gamma_pinv, n, N) - b_ext

def dfnl_dX_blocks(X):
    """
    returns list of N blocks J_j = ∂f_nl/∂X at each time sample
    each block is 2x2
    """
    y = np.sqrt(2.0) * (X[0, :] + X[1, :])
    s = np.sin(y)  # shape (N,)

    # J_j = 2 sin(y_j) * [[beta_a, beta_a],[beta_b, beta_b]]
    base = np.array([[beta_a_p, beta_a_p],
                     [beta_b_p, beta_b_p]])  # 2x2

    J_blocks = [(2.0 * s[j]) * base for j in range(X.shape[1])]
    return J_blocks

def build_dftilde_dx_tilde(X, n, N):
    """
    Block diagonal (nN x nN) with n×n blocks per time sample,
    consistent with Gamma = kron(Phi, I_n) (time-major stacking):
      tilde x = [x(t1); x(t2); ...], each x(tj) in R^n.
    """
    J_blocks = dfnl_dX_blocks(X)  # length N, each (n,n)
    Jbig = lil_matrix((n*N, n*N), dtype=float)

    for j in range(N):
            Jj = J_blocks[j]
            rows = slice(j*n, (j+1)*n)
            cols = slice(j*n, (j+1)*n)
            Jbig[rows, cols] = Jj

    return Jbig.tocsr()

def db_dz(z, Gamma, Gamma_pinv, n, N):
    # time samples
    x_tilde = Gamma @ z
    X = x_tilde_to_X(x_tilde, n, N)

    # ∂f/∂x in time is block diagonal, J_j = 2 sin(y_j) [[β_a, β_a],[β_b, β_b]]:
    # apply it block by block (vectorised, same result as Gamma_pinv @ (Jbig @ Gamma))
    s = np.sin(np.sqrt(2.0) * (X[0, :] + X[1, :]))
    base = np.array([[beta_a_p, beta_a_p],
                     [beta_b_p, beta_b_p]])
    G = Gamma.reshape(N, n, -1)                       # time-major: row j*n + a
    JG = np.einsum('j,ab,jbm->jam', 2.0 * s, base, G).reshape(n * N, -1)

    # chain rule
    return Gamma_pinv @ JG

def jacobian(z, L, Gamma, Gamma_pinv, n, N):
    return L + db_dz(z, Gamma, Gamma_pinv, n, N)
def newton_solve(z0, L, b_ext, Gamma, Gamma_pinv, n, N, tol=1e-10, maxit=50):
    z = z0.copy()
    for it in range(maxit):
        r = residual(z, L, b_ext, Gamma, Gamma_pinv, n, N)
        norm_r = np.linalg.norm(r)
        if norm_r < tol:
            print("Converged", it, norm_r)
            return z

        J = jacobian(z, L, Gamma, Gamma_pinv, n, N)
        dz = np.linalg.solve(J, -r)

        # optional damping
        z = z + dz
        print(it, norm_r, np.linalg.norm(dz))
    raise RuntimeError("Newton did not converge")

def powell_hybrid_solve(z0, L, b_ext, Gamma, Gamma_pinv, n, N,
                        tol=1e-10, maxit=50, use_jac=True, verbose = True):
    """
    Powell hybrid method (MINPACK hybr) via scipy.optimize.root.
    """

    def fun(z):
        return residual(z, L, b_ext, Gamma, Gamma_pinv, n, N)

    jac = (lambda z: jacobian(z, L, Gamma, Gamma_pinv, n, N)) if use_jac else None

    sol = root(
        fun, z0,
        jac=jac,
        method="hybr",
        tol=tol,
        options={"maxfev": maxit * (len(z0) + 1)}
    )

    if not sol.success:
        raise RuntimeError(f"Powell hybrid did not converge: {sol.message}")

    if verbose : print("Converged", sol.nfev, np.linalg.norm(sol.fun))
    return sol.x


In [ ]:
n   = 2 # number of modes
N_H = 8 # number of harmonics
# Number of instant for AFT
N = 900*N_H
# Uniform grid
omega = 4.0
nu = 4
tau_j = np.linspace(0, 100*np.pi, N)

z_0 = np.pad(z_0_init, (0,n*(2*N_H+1)-len(z_0_init)), constant_values=0.0)

M = np.array([[1/4, 0.0],
              [0.0, 1/16]])
C = np.array([[0.0, 0.0],
              [0.0, 0.5*kappa_b_p]]) 
K = np.eye(2)
L = build_L(M, C, K, N_H, omega, nu)
Gamma = build_Gamma(tau_j, N_H, n, omega, nu)
Gamma_inverse = np.linalg.pinv(Gamma)
b_ext = build_b_ext(N_H, eps_d_p, nu)

In [ ]:
#z_sol = newton_solve(z_0, L, b_ext, Gamma, Gamma_inverse, n, N, maxit=50)
# z_0_init was converged with the former g2 (J1): continue it in eps_d up to epsilon_d = 10.12 g2
for eps_d_s in np.linspace(10.12*g2_J1_of(epsilon_p), epsilon_d, 25):
    z_0 = powell_hybrid_solve(z_0, L, build_b_ext(N_H, eps_d_s*phi_b/w_b, nu), Gamma, Gamma_inverse, n, N, verbose=False)
z_sol = powell_hybrid_solve(z_0, L, b_ext, Gamma, Gamma_inverse, n, N)

In [ ]:
def plot_phase_space(z_solution, N_H):
    """
    z_solution: array of length 2*(2*N_H+1)
    """

    # time grid (one period)
    t = np.linspace(0, 100, 1000)

    # split DOFs
    za = z_solution[::2]        # memory
    zb = z_solution[1::2]        # buffer

    # initialize
    x_a = np.zeros_like(t)
    p_a = np.zeros_like(t)
    x_b = np.zeros_like(t)
    p_b = np.zeros_like(t)

    # DC terms
    x_a += za[0] / np.sqrt(2)
    x_b += zb[0] / np.sqrt(2)

    # harmonics
    for k in range(1, N_H + 1):
        s_idx = 1 + 2*(k-1)
        c_idx = 2 + 2*(k-1)

        # memory (a)
        x_a += za[s_idx]*np.sin(k*t) + za[c_idx]*np.cos(k*t)
        p_a += k/2*za[s_idx]*np.cos(k*t) - k/2*za[c_idx]*np.sin(k*t)

        # buffer (b)
        x_b += zb[s_idx]*np.sin(k*t) + zb[c_idx]*np.cos(k*t)
        p_b += k/4*zb[s_idx]*np.cos(k*t) - k/4*zb[c_idx]*np.sin(k*t)

    a = (x_a + 1j*p_a)/np.sqrt(2)
    a_R = a*np.exp(2j*t)

    b = (x_b + 1j*p_b)/np.sqrt(2)
    b_R = b*np.exp(4j*t)

    # ---- plots ----
    plt.figure()
    plt.plot(np.real(a_R), np.imag(a_R))
    plt.xlabel(r"$\Re(a_R)$")
    plt.ylabel(r"$\Im(a_R)$")
    plt.title("Phase portrait: memory (a)")
    plt.grid(True)
    plt.gca().set_aspect('equal')

    plt.figure()
    plt.plot(np.real(b_R), np.imag(b_R))
    plt.xlabel(r"$\Re(b_R)$")
    plt.ylabel(r"$\Im(b_R)$")
    plt.title("Phase portrait: buffer (b)")
    plt.grid(True)
    plt.gca().set_aspect('equal')
    plt.show()

In [ ]:
z_sol

In [ ]:
plot_phase_space(z_sol, N_H)

In [ ]:
Re_a_list = [-0.3888888888888889]#np.linspace(-0.5,0.5,10)#
Im_a_list = [-0.16666666666666669]#np.linspace(-0.5,0.5,10)#
n   = 2 # number of modes
N_H = 8 # number of harmonics
# Number of instant for AFT
N = 900*N_H
# Uniform grid
omega = 4.0
nu = 4
tau_j = np.linspace(0, 100*np.pi, N)

M = np.array([[1.0/4, 0.0],
              [0.0, 1/16]])
C = np.array([[0.0, 0.0],
              [0.0, 0.5*kappa_b_p]]) 
K = np.eye(n)
L = build_L(M, C, K, N_H, omega, nu)
Gamma = build_Gamma(tau_j, N_H, n, omega, nu)
Gamma_inverse = np.linalg.pinv(Gamma)
b_ext = build_b_ext(N_H, eps_d_p, nu)
z_sol_list=[]
for re_a in Re_a_list:
    print(re_a)
    for im_a in Im_a_list:
        z_0_init = np.array([ 
            1.67375389e-01,  2.87830339e-01,
            re_a, im_a, re_a, im_a,
            6.92620826e-01, 0.0, im_a,  0.0,
            re_a, im_a, re_a, im_a,
            0.0, 0.0, 0.0,  5.69644528e-01,
            re_a, im_a, re_a, im_a,
            -9.22066305e-03, im_a, -1.18157915e-03, re_a,
            ])
        z_0 = np.pad(z_0_init, (0,n*(2*N_H+1)-len(z_0_init)), constant_values=0.01)
        
        try:
            #z_sol = newton_solve(
            #    z_0, L, b_ext, Gamma, Gamma_inverse,
            #    n, N, maxit=50
            #)
            z_sol = powell_hybrid_solve(z_0, L, b_ext, Gamma, Gamma_inverse, n, N, verbose = False)
            print(re_a, im_a)
            z_sol_list.append(z_sol)
        except Exception as e:
            print(e)
            pass

In [ ]:
z_sol_list

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax_a, ax_b = axes

for z_sol in z_sol_list:

    if z_sol is None:
        continue

    # one HB period
    t = np.linspace(0, 2*np.pi, 400)

    # split DOFs (time-major Fourier ordering)
    za = z_sol[::2]   # mode a
    zb = z_sol[1::2]  # mode b

    # initialize
    x_a = np.zeros_like(t)
    p_a = np.zeros_like(t)
    x_b = np.zeros_like(t)
    p_b = np.zeros_like(t)

    # DC
    x_a += za[0] / np.sqrt(2)
    x_b += zb[0] / np.sqrt(2)

    # harmonics
    for k in range(1, N_H + 1):
        s_idx = 1 + 2*(k-1)
        c_idx = 2 + 2*(k-1)

        # mode a
        x_a += za[s_idx]*np.sin(k*t) + za[c_idx]*np.cos(k*t)
        p_a += k/2*za[s_idx]*np.cos(k*t) - k/2*za[c_idx]*np.sin(k*t)

        # mode b  (remember physical frequency factor 1/2 in your model)
        x_b += zb[s_idx]*np.sin(k*t) + zb[c_idx]*np.cos(k*t)
        p_b += (k/4)*zb[s_idx]*np.cos(k*t) - (k/4)*zb[c_idx]*np.sin(k*t)

    # complex amplitudes
    a = (x_a + 1j*p_a)/np.sqrt(2)
    b = (x_b + 1j*p_b)/np.sqrt(2)

    a = (x_a + 1j*p_a)/np.sqrt(2)
    a_R = a*np.exp(2j*t)

    b = (x_b + 1j*p_b)/np.sqrt(2)
    b_R = b*np.exp(4j*t)

    ax_a.plot(np.real(a_R), np.imag(a_R))
    ax_b.plot(np.real(b_R), np.imag(b_R))

# formatting
ax_a.set_title("Phase portrait: mode a")
ax_a.set_xlabel(r"$\Re(a_R)$")
ax_a.set_ylabel(r"$\Im(a_R)$")
ax_a.set_xlim([-1,1])
ax_a.set_ylim([-1,1])
ax_a.set_aspect('equal')
ax_a.grid(True)

ax_b.set_title("Phase portrait: mode b")
ax_b.set_xlabel(r"$\Re(b_R)$")
ax_b.set_ylabel(r"$\Im(b_R)$")
#ax_b.set_xlim([-.2,.5])
#ax_b.set_ylim([-.2,.2])
ax_b.set_aspect('equal')
ax_b.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
za_1_list=[]
for z_sol in z_sol_list:
    if z_sol is None:
        continue
    za = z_sol[::2]   # mode a
    za_1 = [round(coef, 2) for coef in za[3:5]]
    za_1_list.append(za_1)

In [ ]:
1/np.sqrt(2)

In [ ]:
np.unique(np.array(za_1_list), axis = 0)

# Checks

In [ ]:
import numpy as np

def build_Phi_terms(t, N_H):
    """Return Phi, dPhi, ddPhi for basis [1/sqrt2, sin, cos, ...]"""
    t = np.asarray(t)
    N = t.size
    Phi = np.zeros((N, 2*N_H+1))
    dPhi = np.zeros_like(Phi)
    ddPhi = np.zeros_like(Phi)

    Phi[:, 0] = 1/np.sqrt(2)

    for k in range(1, N_H+1):
        is_ = 1 + 2*(k-1)
        ic_ = 2 + 2*(k-1)

        Phi[:, is_] = np.sin(k*t)
        Phi[:, ic_] = np.cos(k*t)

        dPhi[:, is_] =  k*np.cos(k*t)
        dPhi[:, ic_] = -k*np.sin(k*t)

        ddPhi[:, is_] = -(k**2)*np.sin(k*t)
        ddPhi[:, ic_] = -(k**2)*np.cos(k*t)

    return Phi, dPhi, ddPhi

def build_Gamma_from_Phi(Phi, n):
    return np.kron(Phi, np.eye(n))

def check_L_vs_Gamma(L, t, N_H, n, M, C, K, trials=5, tol=1e-10):
    N = len(t)
    Phi, dPhi, ddPhi = build_Phi_terms(t, N_H)

    Gamma   = build_Gamma_from_Phi(Phi, n)
    Gamma_d = build_Gamma_from_Phi(dPhi, n)
    Gamma_dd= build_Gamma_from_Phi(ddPhi, n)

    # time-domain linear operator on stacked vectors
    Mbig = np.kron(np.eye(N), M)  # (nN x nN) but note stacking convention!
    Cbig = np.kron(np.eye(N), C)
    Kbig = np.kron(np.eye(N), K)

    # IMPORTANT: above assumes time-major stacking [x(t1); x(t2); ...]
    # If your tilde x is DOF-major [x1(t1..tN); x2(t1..tN)], use this instead:
    #Mbig = np.kron(M, np.eye(N))
    #Cbig = np.kron(C, np.eye(N))
    #Kbig = np.kron(K, np.eye(N))

    for r in range(trials):
        z = np.random.randn(n*(2*N_H+1))

        x_tilde   = Gamma   @ z
        dx_tilde  = Gamma_d @ z
        ddx_tilde = Gamma_dd@ z

        # time-domain linear result
        lin_time = Mbig @ ddx_tilde + Cbig @ dx_tilde + Kbig @ x_tilde

        # Fourier-domain linear result mapped back to time
        lin_freq_time = Gamma @ (L @ z)

        err = np.linalg.norm(lin_time - lin_freq_time) / max(1.0, np.linalg.norm(lin_time))
        print(f"trial {r}: relative error = {err:.3e}")

        if err > tol:
            print("❌ Inconsistent (likely ordering or sign/scaling mismatch).")
            return False

    print("✅ L is consistent with Gamma (within tolerance).")
    return True


In [ ]:
n = 2
N_H = 5
N = 16*N_H
t = np.linspace(0, 2*np.pi, N, endpoint=False)  # or your period

# put your matrices here
M = np.array([[1.0, 0.0],[0.0, 0.25]])
C = np.array([[0.0, 0.0],[0.0, 0.5*kappa_b_p]])
K = np.eye(2)

L = build_L(M, C, K, N_H)  # make sure your build_L uses (M,C,K) in this order

check_L_vs_Gamma(L, t, N_H, n, M, C, K)


# Floquet coefficients

In [ ]:
from scipy.linalg import eig as scipy_eig
def complex_harmonics(U, N_H, n):
    """
    Eigenvectors u (columns of U, ordering [c0, s1, c1, ...]) in the complex basis e^{ik tau}, k = -N_H..N_H:
        c_k cos(k t) + s_k sin(k t) = (c_k - i s_k)/2 e^{ikt} + (c_k + i s_k)/2 e^{-ikt}
    Returns an array of shape (2N_H+1, n, ncols), index N_H + k.
    """
    A = np.zeros((2*N_H + 1, n, U.shape[1]), dtype=complex)
    A[N_H] = U[:n, :] / np.sqrt(2)
    for k in range(1, N_H + 1):
        s = U[(2*k - 1)*n:(2*k)*n, :]
        c = U[(2*k)*n:(2*k + 1)*n, :]
        A[N_H + k] = (c - 1j*s) / 2
        A[N_H - k] = (c + 1j*s) / 2
    return A


def harmonic_centroid(A, N_H):
    """
    Mean harmonic index of each eigenvector. A spurious copy lambda + i m of a genuine exponent has its
    eigenvector shifted by -m harmonics, so the genuine ones are centred on k = 0 (Lazarus & Thomas 2010).
    """
    w = np.sum(np.abs(A)**2, axis=1)                 # (2N_H+1, ncols)
    k = np.arange(-N_H, N_H + 1)[:, None]
    return np.sum(k*w, axis=0) / np.sum(w, axis=0)


def shifted_overlap(a, b, m, N_H):
    """|<a shifted by -m harmonics, b>| / (|a| |b|) for single eigenvectors a, b of shape (2N_H+1, n)."""
    if m > 0:
        a_s, b_s = a[m:], b[:-m]
    elif m < 0:
        a_s, b_s = a[:m], b[-m:]
    else:
        a_s, b_s = a, b
    return np.abs(np.vdot(a_s, b_s)) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-300)


def select_genuine(eigvals, eigvecs, N_H, n, tol=1e-3, overlap_min=0.9):
    """
    Keep the 2n genuine Floquet exponents among the 2n(2N_H+1) Hill eigenvalues.

    Each Floquet multiplier appears in the Hill spectrum as a family lambda + i m (m integer), whose
    eigenvectors are shifted copies of each other in harmonic space. Candidates are ranked by
    |harmonic centroid| (then |Im|), and only one member per family is kept. This matters at period
    doubling: a multiplier on the negative real axis has two members lambda = sigma ± i/2 with the same
    |Im|, and Moore's |Im| criterion alone would pick both and drop another multiplier.

    The returned exponents are folded into the first zone |Im| <= 1/2 (Moore's convention, as in the
    paper); the multipliers exp(lambda T) are unchanged. The eigenvectors are those of the centred member.
    """
    U = eigvecs[eigvecs.shape[0]//2:, :]          # v = [lambda u ; u] -> bottom half is u
    A = complex_harmonics(U, N_H, n)
    centroid = harmonic_centroid(A, N_H)
    order = np.lexsort((np.abs(np.imag(eigvals)), np.abs(centroid)))
    chosen = []
    for j in order:
        lam = eigvals[j]
        if not np.isfinite(lam):
            continue
        same_family = False
        for c in chosen:
            d = lam - eigvals[c]
            m = int(np.round(np.imag(d)))
            if (m != 0 and abs(np.real(d)) < tol and abs(np.imag(d) - m) < tol
                    and shifted_overlap(A[:, :, c], A[:, :, j], m, N_H) > overlap_min):
                same_family = True
                break
        if not same_family:
            chosen.append(j)
        if len(chosen) == 2*n:
            break
    chosen = np.array(chosen)
    lam = eigvals[chosen]
    lam_folded = lam - 1j*np.round(np.imag(lam))
    # a real negative multiplier has members sigma ± i/2: always report it with Im = +1/2
    lam_folded = np.where(np.abs(np.imag(lam_folded) + 0.5) < 1e-6, lam_folded + 1j, lam_folded)
    return lam_folded, eigvecs[:, chosen]


def liouville_residual(eigvals, M, C):
    """
    Liouville/Abel: product of the Floquet multipliers = exp(-tr(M^-1 C) T), i.e. the real parts of
    the 2n genuine exponents must sum to -tr(M^-1 C). Returns the mismatch (should be ~0).
    """
    return np.sum(np.real(eigvals)) + np.trace(np.linalg.solve(M, C))


def Floq_coef(z_sol, L, M, C, N_H, n):

    Delta_1 = np.zeros([(2*N_H + 1)*n, (2*N_H + 1)*n])

    # Put C on every diagonal block (including k=0)
    for i in range(2*N_H + 1):
        Delta_1[i*n:(i+1)*n, i*n:(i+1)*n] = C

    # Put ±2k M between the (sin,cos) blocks for each k=1..N_H
    for k in range(1, N_H + 1):
        i = 2*k - 1  # sin index
        j = 2*k      # cos index

        # upper diagonal (sin row, cos column)
        Delta_1[i*n:(i+1)*n, j*n:(j+1)*n] = -2*k * M

        # lower diagonal (cos row, sin column)
        Delta_1[j*n:(j+1)*n, i*n:(i+1)*n] =  2*k * M


    Delta_2 = block_diag(*([M] * (2*N_H + 1)))
    h_z = jacobian(z_sol, L, Gamma, Gamma_inverse, n, N)

    B_1 = np.block([[Delta_1, h_z],[-np.eye((2*N_H + 1)*n),np.zeros([(2*N_H + 1)*n, (2*N_H + 1)*n])]])
    B_2 = -np.block([[Delta_2, np.zeros([(2*N_H + 1)*n, (2*N_H + 1)*n])],[np.zeros([(2*N_H + 1)*n, (2*N_H + 1)*n]), np.eye((2*N_H + 1)*n)]])
    eigvals, eigvecs = scipy_eig(B_1, B_2)

    # keep the 2n genuine Floquet exponents (Detroux et al. 2015, Sec. 3.1)
    return select_genuine(eigvals, eigvecs, N_H, n)

In [ ]:
from scipy.optimize import linear_sum_assignment

def match_eigs(prev_eigvals, prev_eigvecs, new_eigvals, new_eigvecs,
                alpha=1.0, beta=5.0, T=2*np.pi, n_dof=2):
    """
    Match new eigenpairs to previous ones using both eigenvalue proximity
    and eigenvector overlap (subspace continuity). Returns a permutation
    array 'perm' such that new_eigvals[perm] aligns with prev_eigvals.

    Both terms are insensitive to the i*m ambiguity of Floquet exponents, which otherwise makes the
    branches swap after a period doubling (a real negative multiplier sits at Im(lambda) = ±1/2):
      - proximity is measured between multipliers mu = exp(lambda T), not exponents;
      - overlap is maximised over harmonic shifts m = -1, 0, 1 (members of a family are shifted copies).
    """
    n = len(prev_eigvals)
    cost = np.zeros((n, n))

    n_rows = prev_eigvecs.shape[0] // 2               # v = [lambda u ; u] -> bottom half is u
    N_H_v = (n_rows // n_dof - 1) // 2
    A_prev = complex_harmonics(prev_eigvecs[n_rows:, :], N_H_v, n_dof)
    A_new  = complex_harmonics(new_eigvecs[n_rows:, :], N_H_v, n_dof)

    mu_prev, mu_new = np.exp(prev_eigvals*T), np.exp(new_eigvals*T)
    mu_scale = np.max(np.abs(mu_prev)) + 1e-12

    for i in range(n):
        for j in range(n):
            dval     = np.abs(mu_prev[i] - mu_new[j]) / mu_scale
            overlap  = max(shifted_overlap(A_prev[:, :, i], A_new[:, :, j], m, N_H_v) for m in (-1, 0, 1))  # in [0,1]
            dvec     = 1.0 - overlap
            cost[i, j] = alpha * dval + beta * dvec

    row_ind, perm = linear_sum_assignment(cost)
    # row_ind is just 0..n-1 in order since cost is square; perm gives new->prev mapping
    return perm

In [ ]:
z_guess = np.array([-8.67480681e-02, -1.49178000e-01, -1.55487349e-04, -3.92476662e-06,
        4.80123540e-02,  6.60523141e-02, -3.00220194e-02,  3.06241505e-16,
        3.31363194e-01, -1.81473366e-16, -3.88737312e-03,  1.57262610e-02,
        3.35330370e-02, -1.65080648e-01,  6.93437379e-04, -5.59462961e-02,
       -4.67601081e-03,  3.00083819e-01,  4.52017869e-05,  9.32575051e-04,
       -4.88538089e-04, -7.81645147e-03, -1.26778042e-03, -1.46772441e-02,
        4.61868963e-03,  5.06227939e-02,  2.68514538e-04,  2.61445320e-03,
       -1.01232565e-03, -9.46918403e-03, -5.89963411e-04, -5.17649744e-03,
        1.52354566e-03,  1.30588333e-02])

eps_p_list =np.linspace(1.2, 0.7, 10, endpoint=False)
"""
np.concatenate((
    np.linspace(1.2, 0.7, 10, endpoint=False),
    np.linspace(0.7, 0.55, 50)
))
"""
eigvals_list = []
prev_eigvals = None  # track last successful eigenvalues for adiabatic sorting
n   = 2 # number of modes
N_H = 12 # number of harmonics
# Number of instant for AFT
N = 900*N_H
# Uniform grid
omega = 4.0
nu = 4
tau_j = np.linspace(0, 100*np.pi, N)

z_guess = np.pad(z_guess, (0,n*(2*N_H+1)-len(z_guess)), constant_values=0.0)

# z_guess was converged with the former g2 (J1): continue it in eps_d, at the first eps_p, up to 10.12 g2
varphi_a, varphi_b = np.sqrt(Z_a)/np.sqrt(2)/phi_quantum, np.sqrt(Z_b)/np.sqrt(2)/phi_quantum
beta_a_p, beta_b_p = 2*E_J*np.sin(eps_p_list[0])*varphi_a**2/(w_a), 2*E_J*np.sin(eps_p_list[0])*varphi_b**2/(w_b)
B = np.array([beta_a_p, beta_b_p])
M = np.array([[1/4, 0.0],
            [0.0, 1/16]])
C = np.array([[0.0, 0.0],
            [0.0, 0.5*kappa_b / w_b]])
L = build_L(M, C, np.eye(2), N_H, omega, nu)
Gamma = build_Gamma(tau_j, N_H, n, omega, nu)
Gamma_inverse = np.linalg.pinv(Gamma)
for eps_d_s in np.linspace(10.12*g2_J1_of(eps_p_list[0]), 10.12*g2_of(eps_p_list[0]), 25):
    z_guess = powell_hybrid_solve(z_guess, L, build_b_ext(N_H, eps_d_s*varphi_b/w_b, nu), Gamma, Gamma_inverse, n, N, verbose=False)
for epsilon_p in eps_p_list[::]:
    print(epsilon_p)
    g2 = g2_of(epsilon_p)
    print(g2/2/np.pi*1e3)
    epsilon_d = 10.12 *g2#0.75*g2_max

    varphi_a, varphi_b = np.sqrt(Z_a)/np.sqrt(2)/phi_quantum, np.sqrt(Z_b)/np.sqrt(2)/phi_quantum
    beta_a_p, beta_b_p = 2*E_J*np.sin(epsilon_p)*varphi_a**2/(w_a), 2*E_J*np.sin(epsilon_p)*varphi_b**2/(w_b)
    B = np.array([beta_a_p, beta_b_p])  # shape (2,)
    eps_d_p = epsilon_d*varphi_b/w_b
    kappa_b_p = kappa_b / w_b

    M = np.array([[1/4, 0.0],
                [0.0, 1/16]])
    C = np.array([[0.0, 0.0],
                [0.0, 0.5*kappa_b_p]]) 
    K = np.eye(2)
    L = build_L(M, C, K, N_H, omega, nu)
    Gamma = build_Gamma(tau_j, N_H, n, omega, nu)
    Gamma_inverse = np.linalg.pinv(Gamma)
    b_ext = build_b_ext(N_H, eps_d_p, nu)
    try :
        z_sol = powell_hybrid_solve(z_guess, L, b_ext, Gamma, Gamma_inverse, n, N, verbose = False)
        plot_phase_space(z_sol, N_H)
        new_eigvals, new_eigvecs = Floq_coef(z_sol, L, M, C, N_H, n)

        if prev_eigvals is None:
            sorted_eigvals = new_eigvals
            sorted_eigvecs = new_eigvecs
        else:
            perm = match_eigs(prev_eigvals, prev_eigvecs, new_eigvals, new_eigvecs, T=2*np.pi*nu/omega)
            sorted_eigvals = new_eigvals[perm]
            sorted_eigvecs = new_eigvecs[:, perm]

        eigvals_list.append(sorted_eigvals)
        prev_eigvals = sorted_eigvals
        prev_eigvecs = sorted_eigvecs
        z_guess      = z_sol

    except Exception as e:
        print(f"Failed at epsilon_p={epsilon_p:.4f}: {e}")
        # Don't update z_guess or prev_eigvals — keep last good state
        eigvals_list.append(np.full(2*n, np.nan + 1j * np.nan))
eigvals_array = np.array(eigvals_list)
print("max Liouville residual:", np.nanmax([abs(liouville_residual(ev, M, C)) for ev in eigvals_array]))


In [ ]:
T=2*np.pi/(omega/4) # Period corresponding to the memory frequency (one fourth of the drive period)
Poincare_eig = np.exp(eigvals_array*T)

In [ ]:
Poincare_eig.T.shape

In [ ]:
plt.figure()
for i in range(2*n):
    plt.plot(eps_p_list[:], np.real(eigvals_array[:, i]))
plt.xlabel("Pump amplitude "+r"$\varepsilon_p$")
plt.ylabel("Real part of Floquet\n exponents")
plt.show()

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
import Plot_setting
from Plot_setting import *
plt.figure()
for i in range(2*n):
    plt.plot(eps_p_list[:], np.imag(eigvals_array[:, i]))
plt.xlabel("Pump amplitude "+r"$\varepsilon_p$")
plt.ylabel("Imaginary part of Floquet\n exponents")
plt.show()

In [ ]:
g2_list = g2_of(eps_p_list)
plt.figure()
for i in range(2*n):
    plt.plot(g2_list[-40:]/2/np.pi*1e3, np.real(Poincare_eig[:, i])[-40:])
plt.axhline([1], ls="--", color="r")
plt.xlabel(r"$g_2/2\pi$"+" (MHz)")
plt.ylabel("Real part of Floquet \n multipliers")
plt.tight_layout()
#plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/poster_chicago/floquet_multiplier_after_period_doubling_08_10_2026.svg")
plt.show()

In [ ]:
# Store what is plotted above (and the exponents), to replot in plot_floquet_multipliers.ipynb
import json
floquet_data = {
    "description": "Floquet exponents and multipliers of the period-2 (satellite) orbit vs eps_p, from Hill_method_doubling_corrected.ipynb",
    "eps_d_over_g2": 10.12,          # value used in the sweep cell
    "g2_definition": "E_J sin(eps_p) phi_a^2 phi_b",
    "kappa_b": kappa_b,
    "N_H": N_H,
    "T": T,
    "eps_p": eps_p_list.tolist(),
    "g2_MHz": (g2_list/2/np.pi*1e3).tolist(),
    "exponents_real": np.real(eigvals_array).tolist(),     # shape (len(eps_p), 2n), NaN where the solver failed
    "exponents_imag": np.imag(eigvals_array).tolist(),
    "multipliers_real": np.real(Poincare_eig).tolist(),
    "multipliers_imag": np.imag(Poincare_eig).tolist(),
}
with open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/floquet_after_period_doubling_12_10_2026.json", "w") as f:
    json.dump(floquet_data, f)

In [ ]:
plt.figure()
for i in range(2*n):
    plt.plot(eps_p_list[:], np.abs(Poincare_eig[:, i]))
#plt.axhline([-1], ls="--", color="r")
plt.xlabel("Pump amplitude "+r"$\varepsilon_p$")
plt.ylabel("Absolute value of Poincaré \n eigenvalues")
plt.show()

# Loop over the drive amplitude

In [ ]:
z_seed = np.array([-8.67480681e-02, -1.49178000e-01, -1.55487349e-04, -3.92476662e-06,
        4.80123540e-02,  6.60523141e-02, -3.00220194e-02,  3.06241505e-16,
        3.31363194e-01, -1.81473366e-16, -3.88737312e-03,  1.57262610e-02,
        3.35330370e-02, -1.65080648e-01,  6.93437379e-04, -5.59462961e-02,
       -4.67601081e-03,  3.00083819e-01,  4.52017869e-05,  9.32575051e-04,
       -4.88538089e-04, -7.81645147e-03, -1.26778042e-03, -1.46772441e-02,
        4.61868963e-03,  5.06227939e-02,  2.68514538e-04,  2.61445320e-03,
       -1.01232565e-03, -9.46918403e-03, -5.89963411e-04, -5.17649744e-03,
        1.52354566e-03,  1.30588333e-02])  # period-2 orbit at (eps_p = 1.2, eps_d/g2 = 10.12), the seed of the 1D sweep above

n   = 2 # number of modes
N_H = 15 # number of harmonics
# Number of instant for AFT
N = 900*N_H
# Uniform grid
omega = 4.0
nu = 4
tau_j = np.linspace(0, 100*np.pi, N)

eps_d_ratio_list = np.linspace(2, 13.5, 47)   # was linspace(5, 13.5, 40); extended to overlap the OTOC grid
eps_p_list = np.linspace(0.54, 0.78, 73)   # contains the threshold for every eps_d/g2 (about 0.575 at 13.5 to 0.73 at 2)

Nd = len(eps_d_ratio_list)
Np = len(eps_p_list)

# preallocate storage, along the ascending eps_d_ratio_list and eps_p_list
eigvals_array = np.full((Nd, Np, 2*n), np.nan, dtype=complex)
odd_content_array = np.full((Nd, Np), np.nan)   # period-2 content of mode a (0 on the period-1 orbit)
varphi_a = np.sqrt(Z_a)/np.sqrt(2)/phi_quantum
varphi_b = np.sqrt(Z_b)/np.sqrt(2)/phi_quantum
eps_d_p_base = varphi_b/w_b
kappa_b_p = kappa_b / w_b
M = np.array([[1/4, 0.0],
                [0.0, 1/16]])
C = np.array([[0.0, 0.0],
                [0.0, 0.5*kappa_b_p]])
K = np.eye(2)
L_base = build_L(M, C, K, N_H, omega, nu)
Gamma = build_Gamma(tau_j, N_H, n, omega, nu)
Gamma_inverse = np.linalg.pinv(Gamma)


def set_point(epsilon_p, eps_d_ratio):
    """Update the globals used by f_nl_time / db_dz and return b_ext."""
    global B, beta_a_p, beta_b_p
    g2 = g2_of(epsilon_p)
    eps_d_p = eps_d_ratio * g2 * eps_d_p_base
    beta_a_p = 2*E_J*np.sin(epsilon_p)*varphi_a**2/(w_a)
    beta_b_p = 2*E_J*np.sin(epsilon_p)*varphi_b**2/(w_b)
    B = np.array([beta_a_p, beta_b_p])  # shape (2,)
    return build_b_ext(N_H, eps_d_p, nu)


def odd_harmonic_content(z):
    """Norm of the odd harmonics of mode a (frequencies 1/2, 3/2, ... of the memory): zero unless period-doubled."""
    za = z[::2]
    return np.sqrt(sum(za[1 + 2*(k-1)]**2 + za[2 + 2*(k-1)]**2 for k in range(1, N_H + 1, 2)))


def solve(z, epsilon_p, eps_d_ratio):
    return powell_hybrid_solve(z, L_base, set_point(epsilon_p, eps_d_ratio), Gamma, Gamma_inverse, n, N, verbose=False)


# 1) continue the seed in eps_p from 1.2 down to the top of the grid, at eps_d/g2 = 10.12.
#    Jumping straight to eps_p = 0.72 lands on the (stable) period-1 cat orbit instead.
z = np.pad(z_seed, (0, n*(2*N_H+1)-len(z_seed)), constant_values=0.0)
#    The seed was converged with the former g2 (J1): continue it first in eps_d, at eps_p = 1.2, up to 10.12 g2.
for eps_d_ratio in np.linspace(10.12*g2_J1_of(1.2)/g2_of(1.2), 10.12, 25):
    z = solve(z, 1.2, eps_d_ratio)
for epsilon_p in np.linspace(1.2, eps_p_list[-1], 25):
    z = solve(z, epsilon_p, 10.12)

# 2) continue in eps_d/g2 along the top row eps_p = eps_p_list[-1]
i0 = np.argmin(np.abs(eps_d_ratio_list - 10.12))
z_top = [None]*Nd
z_r = z
for i in range(i0, -1, -1):
    z_r = solve(z_r, eps_p_list[-1], eps_d_ratio_list[i])
    z_top[i] = z_r
z_r = z_top[i0]
for i in range(i0 + 1, Nd):
    z_r = solve(z_r, eps_p_list[-1], eps_d_ratio_list[i])
    z_top[i] = z_r

# 3) for each eps_d/g2, continue downward in eps_p
for i, eps_d_ratio in enumerate(eps_d_ratio_list):
    print(eps_d_ratio)
    z_guess = z_top[i]
    for j_rev, epsilon_p in enumerate(eps_p_list[::-1]):
        j = Np - 1 - j_rev  # index in the ascending eps_p_list
        try:
            z_sol = solve(z_guess, epsilon_p, eps_d_ratio)
            eigvals_array[i, j, :], _ = Floq_coef(z_sol, L_base, M, C, N_H, n)
            odd_content_array[i, j] = odd_harmonic_content(z_sol)
            z_guess = z_sol

        except Exception as e:
                print(e)


In [ ]:
T=2*np.pi/(omega/4) # Period corresponding to the memory frequency (one fourth of the drive period)
Poincare_eig = np.exp(eigvals_array[0]*T)  # first eps_d/g2 ratio, ascending eps_p
plt.figure()
for i in range(2*n):
    plt.plot(eps_p_list, np.abs(Poincare_eig[:, i]))
plt.axhline([1], ls="--", color="r")
plt.xlabel("Pump amplitude "+r"$\varepsilon_p$")
plt.ylabel("Absolute value of Floquet \n multipliers")
plt.show()

In [ ]:
T=2*np.pi/(omega/4) # Period corresponding to the memory frequency (one fourth of the drive period)
Poincare_eig = np.exp(eigvals_array*T)
print("max Liouville residual:", np.nanmax(np.abs(np.sum(eigvals_array.real, axis=2) + np.trace(np.linalg.solve(M, C)))))
tol=1e-2
mask = np.any(
    np.abs(Poincare_eig) > 1,
    axis=2
)   # shape (Nd, Np)

epsilon_p_unstable_list = []

for i in range(Nd):
    idx = np.where(mask[i])[0]
    if len(idx) > 0:
        # smallest eps_p at which the period-2 orbit is unstable (eigvals_array is stored along the ascending eps_p_list)
        epsilon_p_unstable_list.append(eps_p_list[idx[0]])
    else:
        epsilon_p_unstable_list.append(np.nan)

epsilon_p_unstable_array = np.array(epsilon_p_unstable_list)
epsilon_p_unstable_array

In [ ]:
plt.figure()
plt.plot(eps_d_ratio_list, g2_of(np.array(epsilon_p_unstable_array))/2/np.pi*1e3)
plt.xlabel(r"$\varepsilon_d/g_2$")
plt.ylabel(r"$g_2^{unstable}/2\pi$ (MHz)")
plt.show()

In [ ]:
import json 
with open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/g2_list_12_10_2026_Hill_unstable.json", "w") as f:
    json.dump(list(g2_of(np.array(epsilon_p_unstable_array))/2/np.pi*1e3), f)   # g2 = E_J sin(eps_p) phi_a^2 phi_b, MHz
# ordered along the ascending eps_d_ratio_list, stored next to it: plot g2 against eps_d_ratio/2, without [::-1]
with open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/eps_d_ratio_list_12_10_2026_Hill_unstable.json", "w") as f:
    json.dump(list(eps_d_ratio_list), f)

# Instability threshold vs $\kappa_b$

At fixed $\varepsilon_d = 10.12\,g_2$: the smallest $\varepsilon_p$ at which the period-2 orbit is unstable (max $|\mu| > 1$),
for $\kappa_b/2\pi$ from $10^{-3}$ to $10^{-1}$ (1/ns, same $-2\kappa_b\dot\phi_b$ convention as the rest of the notebook).

1. The seed orbit ($\varepsilon_p = 1.2$, $\kappa_b = \pi/10.4$) is continued in $\kappa_b$ along $\varepsilon_p = 1.2$.
2. For each $\kappa_b$, $\varepsilon_p$ is decreased in steps of `d_eps_p` until the orbit is stable, then the crossing is refined by bisection.
3. No threshold is recorded (status printed) if the orbit loses its odd harmonics first (it merged back with the period-1 orbit),
   if it is already stable at $\varepsilon_p = 1.2$, or if the solver fails.

In [ ]:
z_seed = np.array([-8.67480681e-02, -1.49178000e-01, -1.55487349e-04, -3.92476662e-06,
        4.80123540e-02,  6.60523141e-02, -3.00220194e-02,  3.06241505e-16,
        3.31363194e-01, -1.81473366e-16, -3.88737312e-03,  1.57262610e-02,
        3.35330370e-02, -1.65080648e-01,  6.93437379e-04, -5.59462961e-02,
       -4.67601081e-03,  3.00083819e-01,  4.52017869e-05,  9.32575051e-04,
       -4.88538089e-04, -7.81645147e-03, -1.26778042e-03, -1.46772441e-02,
        4.61868963e-03,  5.06227939e-02,  2.68514538e-04,  2.61445320e-03,
       -1.01232565e-03, -9.46918403e-03, -5.89963411e-04, -5.17649744e-03,
        1.52354566e-03,  1.30588333e-02])  # period-2 orbit at (eps_p = 1.2, eps_d/g2 = 10.12, kappa_b = pi/10.4)

eps_d_ratio_k = 10.12
kappa_b_ref = np.pi/10.4                              # kappa_b of the seed orbit
kappa_b_list = 2*np.pi*np.linspace(1e-3, .25, 30)        # kappa_b/2pi from 1e-3 to 0.1 (1/ns)
eps_p_top, d_eps_p, eps_p_bottom = 1.2, 0.01, 0.4     # downward sweep in eps_p
n_bisect = 12                                         # bisection steps -> resolution d_eps_p / 2**n_bisect
odd_min = 1e-5                                        # below this, the orbit is no longer period-2

n   = 2 # number of modes
N_H = 15 # number of harmonics
N = 900*N_H
omega = 4.0
nu = 4
tau_j = np.linspace(0, 100*np.pi, N)
Gamma = build_Gamma(tau_j, N_H, n, omega, nu)
Gamma_inverse = np.linalg.pinv(Gamma)
varphi_a = np.sqrt(Z_a)/np.sqrt(2)/phi_quantum
varphi_b = np.sqrt(Z_b)/np.sqrt(2)/phi_quantum
M = np.array([[1/4, 0.0],
              [0.0, 1/16]])


def odd_content_k(z):
    """Norm of the odd harmonics of mode a: zero unless the orbit is period-doubled."""
    za = z[::2]
    return np.sqrt(sum(za[1 + 2*(k-1)]**2 + za[2 + 2*(k-1)]**2 for k in range(1, N_H + 1, 2)))


def point_k(z, epsilon_p, kappa_b, eps_d_ratio=None):
    """Solve the orbit at (eps_p, kappa_b) from the guess z, at eps_d = eps_d_ratio g2 (default eps_d_ratio_k).
    Returns (z_sol, max |mu|, odd-harmonic content)."""
    eps_d_ratio = eps_d_ratio_k if eps_d_ratio is None else eps_d_ratio
    global B, beta_a_p, beta_b_p      # used by f_nl_time / db_dz
    g2 = g2_of(epsilon_p)
    beta_a_p = 2*E_J*np.sin(epsilon_p)*varphi_a**2/(w_a)
    beta_b_p = 2*E_J*np.sin(epsilon_p)*varphi_b**2/(w_b)
    B = np.array([beta_a_p, beta_b_p])
    b_ext = build_b_ext(N_H, eps_d_ratio*g2*varphi_b/w_b, nu)
    C = np.array([[0.0, 0.0],
                  [0.0, 0.5*kappa_b/w_b]])
    L = build_L(M, C, np.eye(2), N_H, omega, nu)
    z_sol = powell_hybrid_solve(z, L, b_ext, Gamma, Gamma_inverse, n, N, verbose=False)
    ev, _ = Floq_coef(z_sol, L, M, C, N_H, n)
    return z_sol, np.max(np.abs(np.exp(ev*2*np.pi*nu/omega))), odd_content_k(z_sol)


# 1) continue the seed in kappa_b along eps_p = eps_p_top
# the seed was converged with the former g2 (J1): continue it first in eps_d, up to eps_d_ratio_k g2
z_ref = np.pad(z_seed, (0, n*(2*N_H+1)-len(z_seed)))
for r in np.linspace(eps_d_ratio_k*g2_J1_of(eps_p_top)/g2_of(eps_p_top), eps_d_ratio_k, 25):
    z_ref, _, _ = point_k(z_ref, eps_p_top, kappa_b_ref, r)
k0 = np.searchsorted(kappa_b_list, kappa_b_ref)
z_top = [None]*len(kappa_b_list)
for ks in (range(k0 - 1, -1, -1), range(k0, len(kappa_b_list))):
    z_k = z_ref
    for k in ks:
        try:
            z_k, _, _ = point_k(z_k, eps_p_top, kappa_b_list[k])
            z_top[k] = z_k
        except Exception as e:
            print(f"continuation in kappa_b stopped at kappa_b/2pi = {kappa_b_list[k]/2/np.pi:.2e}: {e}")
            break

# 2) for each kappa_b, step down in eps_p until the period-2 orbit is stable, then bisect
eps_p_threshold = np.full(len(kappa_b_list), np.nan)
status = [""]*len(kappa_b_list)
for k, kappa_b_k in enumerate(kappa_b_list):
    if z_top[k] is None:
        status[k] = "no seed"
        continue
    try:
        z_u, mu, odd = point_k(z_top[k], eps_p_top, kappa_b_k)
        if mu <= 1:
            status[k] = f"already stable at eps_p = {eps_p_top}"
            continue
        e_u, e_s = eps_p_top, None
        while e_u - d_eps_p >= eps_p_bottom:
            z_n, mu, odd = point_k(z_u, e_u - d_eps_p, kappa_b_k)
            if odd < odd_min:
                status[k] = f"merged with the period-1 orbit at eps_p = {e_u - d_eps_p:.3f} while still unstable"
                break
            if mu <= 1:
                e_s = e_u - d_eps_p
                break
            z_u, e_u = z_n, e_u - d_eps_p
        if e_s is None:
            status[k] = status[k] or f"unstable down to eps_p = {eps_p_bottom}"
            continue
        for _ in range(n_bisect):
            e_m = (e_u + e_s)/2
            z_m, mu, odd = point_k(z_u, e_m, kappa_b_k)
            if mu > 1 and odd >= odd_min:
                z_u, e_u = z_m, e_m
            else:
                e_s = e_m
        eps_p_threshold[k] = (e_u + e_s)/2
        status[k] = "ok"
    except Exception as e:
        status[k] = f"solver failed: {e}"
    print(f"kappa_b/2pi = {kappa_b_k/2/np.pi:.2e}: eps_p threshold = {eps_p_threshold[k]:.4f}  ({status[k]})")

g2_threshold_MHz = g2_of(eps_p_threshold)/2/np.pi*1e3

In [ ]:
coeffs = np.polyfit((kappa_b_list[-20:]/2/np.pi)**2, g2_threshold_MHz[-20:], 1)
m, b = coeffs[0], coeffs[1]


In [ ]:
fig, ax = plt.subplots()
ax.plot((kappa_b_list/2/np.pi*2)**2, g2_threshold_MHz, marker="o", ms=3)
plt.plot((kappa_b_list/2/np.pi*2)**2, m*(kappa_b_list/2/np.pi)**2 + b, '--', label=f'fit: y = {m/2:.0f}x + {b:.0f}')
ax.axvline((kappa_b_ref/2/np.pi*2)**2, ls="--", color="grey", label=r"$\kappa_b = 2\pi/10.4$")
ax.set_xlabel(r"$(\kappa_b/2\pi)^2$ (GHz)")
ax.set_ylabel(r"$g_2^\text{unstable}/2\pi$ (MHz)")
ax.legend()
plt.tight_layout()
#plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/article/instability_vary_kappa_b_12_10_2026.pdf")
#plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/article_chaos/figures_article/instability_vary_kappa_b_12_10_2026.pdf")
plt.show()

# Why does the threshold rise with $\kappa_b$? (report v2, 19/10/2026)

Hypothesis of report v2: the satellite existence condition $2\kappa_b'|y|\le\varepsilon_d'$ (App. "Satellites", phase reduction) gives a threshold linear in $\kappa_b$.

Conventions. `kappa_b` of this notebook is the amplitude rate of $-2\kappa_b\dot\phi_b$, i.e. $\kappa_b^\text{GKSL}/2$. The article uses
$\kappa_b' = \kappa_b^\text{GKSL}/2\omega_b$, which is `kappa_b/w_b` here. $y=|b_0|$ is the rescaled buffer amplitude at $\omega_b$ ($\varphi_b b\to b$).
In the code basis, $x_b^\lambda=\varphi_b x_b$ and $X=\sqrt2(x_a^\lambda+x_b^\lambda)$, so $y$ is the amplitude of harmonic 4 of $x_b^\lambda$ divided by $\sqrt2$.
In physical units, the condition reads $|b_0|\le\varepsilon_d/\kappa_b^\text{GKSL}$: the buffer cannot hold more than a linear resonator driven on resonance.

Three tests, all at $\varepsilon_d/g_2=10.12$ as in the cell above (needs `z_top`, `eps_p_threshold` from it):
1. the ratio $r=2\kappa_b'|y|/\varepsilon_d'$ on the period-2 orbit, along the threshold of Fig. 7. The hypothesis needs $r=1$ there (fold of the satellite);
2. the existence line itself, overlaid on Fig. 7: analytic fold, and numerical continuation of the period-1 satellite in $\kappa_b$;
3. the scaling of the threshold shift with $\kappa_b$.

In [ ]:
kappa_G_list = 2*kappa_b_list                    # GKSL kappa_b (1/ns), x-axis of Fig. 7

def y_of(z):
    # |b_0|: harmonic 4 (w_b) of x_b^lambda = sqrt2 |b_0| cos(...)
    zb = z[1::2]
    return np.hypot(zb[1 + 2*3], zb[2 + 2*3])/np.sqrt(2)

def existence_ratio(z, epsilon_p, kappa_b, eps_d_ratio=eps_d_ratio_k):
    # r = 2 kappa_b' |y| / eps_d', kappa_b' = kappa_b/w_b (notebook kappa_b). The satellite exists for r <= 1.
    eps_d_p = eps_d_ratio*g2_of(epsilon_p)*varphi_b/w_b
    return 2*(kappa_b/w_b)*y_of(z)/eps_d_p

def orbit_at(k, epsilon_p, kappa_b):
    # period-2 orbit at (epsilon_p, kappa_b), continued down in eps_p from z_top[k]
    z, e = z_top[k], eps_p_top
    while e - d_eps_p > epsilon_p:
        e -= d_eps_p
        z, _, _ = point_k(z, e, kappa_b)
    return point_k(z, epsilon_p, kappa_b)

# 1) ratio r along the threshold of Fig. 7
r_thr, y_thr = np.full(len(kappa_b_list), np.nan), np.full(len(kappa_b_list), np.nan)
for k in range(len(kappa_b_list)):
    if np.isnan(eps_p_threshold[k]):
        continue
    z, mu, odd = orbit_at(k, eps_p_threshold[k], kappa_b_list[k])
    y_thr[k] = y_of(z)
    r_thr[k] = existence_ratio(z, eps_p_threshold[k], kappa_b_list[k])
for k in range(0, len(kappa_b_list), 3):
    print(f"kappa_b/2pi = {kappa_G_list[k]/2/np.pi*1e3:5.0f} MHz: g2*/2pi = {g2_threshold_MHz[k]:5.2f} MHz, |y| = {y_thr[k]:.3f}, r = {r_thr[k]:.3f}")

In [ ]:
# 2) existence line of the satellite: fold 2 kappa_b'|y| = eps_d'
from scipy.optimize import brentq

def fold_kappa_G(epsilon_p, eps_d_ratio=eps_d_ratio_k):
    # Leading-order harmonic balance (eq. app_sat_scalar, kappa_b' -> 0 in gamma) at the fold, eps_eff = 0 so y = ybar.
    # Returns the GKSL kappa_b (1/ns) above which the satellite no longer exists: kappa_b' = eps_d'/(2 y), kappa_G = 2 w_b kappa_b'.
    ba = 2*E_J*np.sin(epsilon_p)*varphi_a**2/w_a
    bb = 2*E_J*np.sin(epsilon_p)*varphi_b**2/w_b
    bS = ba + bb
    eps_d_p = eps_d_ratio*g2_of(epsilon_p)*varphi_b/w_b
    def F(yb):
        eta = 2*bb*yb
        u = (np.sqrt((1 + eta)**2 + 4*eta**2) - (1 + eta))/(2*eta)
        x2 = 2*(1 + u)*yb**2/(1 + 2*u)
        return yb*(1 + u) - 2*bS*((1 + (1 + u)**2/2)*yb**2 + (1 + u**2)*x2)
    return w_b*eps_d_p/brentq(F, 1e-3, 10)

eps_p_fold = np.linspace(0.2, 1.0, 60)
kappa_G_fold = np.array([fold_kappa_G(e) for e in eps_p_fold])

# numerical: period-1 satellite (odd harmonics of the period-2 seed removed), continued down in eps_p at kappa_b_ref,
# then up in kappa_b at fixed eps_p until the solver fails (natural continuation stops just before the fold)
def even_only(z):
    z = z.copy()
    for kk in range(1, N_H + 1, 2):
        for off in (1 + 2*(kk-1), 2 + 2*(kk-1)):
            z[2*off:2*off + 2] = 0.0
    return z

k_ref = np.searchsorted(kappa_b_list, kappa_b_ref)
eps_p_p1 = [0.35, 0.4, 0.5, 0.6, 0.7, 0.8]
z_p1 = {}
z, _, _ = point_k(even_only(z_top[k_ref]), eps_p_top, kappa_b_list[k_ref])
for e in np.round(np.arange(eps_p_top, 0.345, -0.01), 3):
    z, mu, odd = point_k(z, e, kappa_b_list[k_ref])
    if e in eps_p_p1:
        z_p1[e] = z
        print(f"period-1 satellite at eps_p = {e}: odd content = {odd:.1e}, |y| = {y_of(z):.3f}")

kappa_G_p1, r_p1 = [], []
for e in eps_p_p1:
    z, kap, dk = z_p1[e], kappa_b_list[k_ref], 0.02
    while dk > 1e-4:
        try:
            z, _, _ = point_k(z, e, kap + dk)
            kap += dk
        except Exception:
            dk /= 2
    kappa_G_p1.append(2*kap)
    r_p1.append(existence_ratio(z, e, kap))
    print(f"eps_p = {e} (g2/2pi = {g2_of(e)/2/np.pi*1e3:.1f} MHz): last converged kappa_b/2pi = {2*kap/2/np.pi*1e3:.0f} MHz "
          f"(analytic fold {fold_kappa_G(e)/2/np.pi*1e3:.0f} MHz), r = {r_p1[-1]:.2f}")
kappa_G_p1 = np.array(kappa_G_p1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(2*width_in, height_in))
ax = axes[0]
ax.plot(kappa_G_list/2/np.pi*1e3, g2_threshold_MHz, marker="o", ms=3, label="Hill threshold (Fig. 7)")
ax.plot(kappa_G_fold/2/np.pi*1e3, g2_of(eps_p_fold)/2/np.pi*1e3, "--", label=r"existence, analytic ($2\kappa_b'|y|=\varepsilon_d'$)")
ax.plot(kappa_G_p1/2/np.pi*1e3, g2_of(np.array(eps_p_p1))/2/np.pi*1e3, "s", ms=4, label="existence, numerical")
ax.axvline(kappa_b_ref/np.pi*1e3, ls=":", color="grey")
ax.set_xlim(0, 520)
ax.set_xlabel(r"$\kappa_b/2\pi$ (MHz)")
ax.set_ylabel(r"$g_2/2\pi$ (MHz)")
ax.legend(fontsize=6)
ax = axes[1]
ax.plot(kappa_G_list/2/np.pi*1e3, r_thr, marker="o", ms=3)
ax.axhline(1, ls="--", color="grey")
ax.set_xlabel(r"$\kappa_b/2\pi$ (MHz)")
ax.set_ylabel(r"$2\kappa_b'|y|/\varepsilon_d'$ at threshold")
plt.tight_layout()
#plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/article/instability_vary_kappa_b_existence_19_10_2026.pdf")
plt.show()

**Result of tests 1–2: the hypothesis does not hold.**
* Along the threshold, $r$ grows from 0.005 to 0.78 instead of staying at 1: the period-2 orbit becomes unstable while the satellite is far from its fold.
  $|y|$ changes little (0.42 → 0.31), so $r$ is roughly $\propto\kappa_b$ and is simply the dissipative tilt $\sin2\theta$.
* The existence line lies below the threshold everywhere in Fig. 7, by 30 MHz at $\kappa_b/2\pi=132$ MHz (satellite still found at $g_2/2\pi=47$ MHz, instability at 77 MHz)
  and by 12 MHz at 413 MHz (78 vs 89 MHz). It does not have the shape of the threshold either: the fold scales as $\kappa_b^\text{fold}\propto\beta_\Sigma\varepsilon_d'\propto g_2^2$,
  i.e. $g_2^\text{exist}\propto\sqrt{\kappa_b}$ (concave, nearly straight over this window), while the threshold is convex.
* The analytic fold lies 12–36% above the numerical one in $\kappa_b$ (leading-order expansion, $|y|\approx0.3$–$0.75$ is not small).
  The numerical points are the last converged $\kappa_b$ ($r=0.87$–$0.95$), so the true fold is slightly above them.

In [ ]:
# 3) scaling of the threshold shift: local exponent of g2*(kappa) - g2*(kappa -> 0)
kG_MHz = kappa_G_list/2/np.pi*1e3
dg2 = g2_threshold_MHz - g2_threshold_MHz[0]
for sl in (slice(1, 8), slice(8, 20), slice(20, 30)):
    p = np.polyfit(np.log(kG_MHz[sl]), np.log(dg2[sl]), 1)[0]
    print(f"kappa_b/2pi in [{kG_MHz[sl][0]:.0f}, {kG_MHz[sl][-1]:.0f}] MHz: d(log dg2)/d(log kappa_b) = {p:.2f}")
c_quad = np.polyfit(kG_MHz**2, g2_threshold_MHz, 1)
print(f"g2*/2pi = {c_quad[1]:.2f} MHz + {c_quad[0]:.3e} (kappa_b/2pi [MHz])^2, "
      f"max residual {np.max(np.abs(np.polyval(c_quad, kG_MHz**2) - g2_threshold_MHz)):.2f} MHz")

In [ ]:
# Conservative limit: multipliers of the period-2 orbit at the smallest kappa_b of the sweep (kappa_b/2pi = 2 MHz)
def floquet_exponents(z, epsilon_p, kappa_b):
    global B, beta_a_p, beta_b_p
    beta_a_p = 2*E_J*np.sin(epsilon_p)*varphi_a**2/w_a
    beta_b_p = 2*E_J*np.sin(epsilon_p)*varphi_b**2/w_b
    B = np.array([beta_a_p, beta_b_p])
    C = np.array([[0.0, 0.0], [0.0, 0.5*kappa_b/w_b]])
    ev, _ = Floq_coef(z, build_L(M, C, np.eye(2), N_H, omega, nu), M, C, N_H, n)
    return ev*w_a/2                                   # 1/ns (tau = w_a t/2)

z = z_top[0]
for e in np.arange(eps_p_top - d_eps_p, 0.8, -d_eps_p):
    z, _, _ = point_k(z, e, kappa_b_list[0])
eps_p_H = np.arange(0.8, 0.5815, -0.0025)
sigma_H, mu_H = [], []
for e in eps_p_H:
    z, _, _ = point_k(z, e, kappa_b_list[0])
    ev = floquet_exponents(z, e, kappa_b_list[0])
    sigma_H.append(np.max(ev.real))
    mu_H.append(np.sort(np.abs(np.exp(ev*4*np.pi/w_a))))
sigma_H, mu_H, g2_H = np.array(sigma_H), np.array(mu_H), g2_of(eps_p_H)/2/np.pi*1e3
for i in range(len(eps_p_H) - 1, -1, -12):
    print(f"g2/2pi = {g2_H[i]:.2f} MHz: |mu| = {np.round(mu_H[i], 4)}, sigma_H = {sigma_H[i]:.3f} /ns")

m_sq = sigma_H > 0.15
a_sq, b_sq = np.polyfit(g2_H[m_sq], sigma_H[m_sq]**2, 1)
print(f"sigma_H^2 = {a_sq:.3f} (g2/2pi - {-b_sq/a_sq:.2f} MHz) /ns^2")

# growth rate of the conservative orbit at the damped threshold, in units of kappa_b^GKSL
order = np.argsort(g2_H)
w_thr = np.sqrt(np.interp(g2_threshold_MHz, g2_H[order], np.clip(sigma_H[order], 0, None)**2))/kappa_G_list
for k in range(5, len(kappa_b_list), 4):
    print(f"kappa_b/2pi = {kG_MHz[k]:5.0f} MHz: sigma_H(g2*) / kappa_b = {w_thr[k]:.2f}")

fig, ax = plt.subplots()
ax.plot(g2_H, sigma_H, label=r"$\sigma_H$, $\kappa_b\to0$")
ax.plot(g2_threshold_MHz, kappa_G_list/2, "o", ms=3, label=r"$\kappa_b/2$ at $g_2^*(\kappa_b)$")
ax.set_xlabel(r"$g_2/2\pi$ (MHz)")
ax.set_ylabel(r"rate (ns$^{-1}$)")
ax.legend()
plt.tight_layout()
plt.show()

**Result of test 3: the shift is quadratic in $\kappa_b$, and it comes from a Krein collision of the conservative orbit.**
* $\Delta g_2^*\propto\kappa_b^{p}$ with $p=2.0$–$2.2$ over the whole range, and $g_2^*/2\pi\simeq75.8+7.9\times10^{-5}(\kappa_b/2\pi)^2$ MHz ($\kappa_b/2\pi$ in MHz) fits all 30 points within 0.4 MHz.
  The "linear at large $\kappa_b$" of the article is the local tangent of this parabola (the dashed fit of Fig. 7 uses the last 6 points only).
* For $\kappa_b\to0$, the period-2 orbit becomes unstable at $g_2/2\pi\simeq76.2$ MHz through a quartet $\mu,\mu^*,1/\mu,1/\mu^*$ leaving the unit circle
  (the moduli come in pairs $|\mu|$, $1/|\mu|$): a Hamiltonian–Hopf (Krein) collision, the conservative counterpart of the Neimark–Sacker bifurcation.
  Its growth rate follows $\sigma_H^2\propto g_2-g_{2,0}$, the generic square root of an eigenvalue collision.
* With damping, the orbit stays stable until $\sigma_H$ exceeds a rate of order $\kappa_b$: $\sigma_H(g_2^*)\approx(0.47$–$0.62)\,\kappa_b^\text{GKSL}$, i.e. close to the amplitude decay rate $\kappa_b/2$ of the buffer.
  With $\sigma_H^2=a(g_2-g_{2,0})$ and $w=\sigma_H(g_2^*)/\kappa_b$, this gives $g_2^*-g_{2,0}\approx w^2\kappa_b^2/a$: quadratic, as observed.

Possible text for Sec. "Increasing the single photon loss": *The instability threshold increases quadratically with $\kappa_b$. Without loss, the period-2 satellite loses stability through a Krein collision of its Floquet multipliers, with a growth rate $\sigma\propto\sqrt{g_2-g_{2,0}}$. The buffer loss damps this growth at a rate of order $\kappa_b/2$, so that the threshold moves by $\Delta g_2\propto\kappa_b^2$.*

In [ ]:
# Floquet multipliers of the period-2 orbit in the complex plane, from a bit above the threshold down to the end of the branch
# (period doubling of the satellite, or a fold at large kappa_b),
# for four values of kappa_b. Black circles: threshold of Fig. 7. Uses floquet_exponents (cell above) and z_top, eps_p_threshold (kappa_b sweep).
k_plot = [0, 5, 17, 29]                                   # kappa_b/2pi = 2, 88, 294, 500 MHz (GKSL)

def continue_in_eps_p(z, e_from, e_to, kappa_b, step=d_eps_p):
    # natural continuation in eps_p with step halving when the solver fails
    e = e_from
    while e > e_to + 1e-12:
        e_next = max(e - step, e_to)
        try:
            z, _, _ = point_k(z, e_next, kappa_b)
            e = e_next
        except Exception:
            step /= 2
            if step < 1e-5:
                raise
    return z

mu_plot = {}
for k in k_plot:
    e_start = max(0.70, eps_p_threshold[k] + 0.03)       # start a bit above the threshold
    z = continue_in_eps_p(z_top[k], eps_p_top, e_start, kappa_b_list[k])
    g2_k, mu_k = [], []
    for e in np.arange(e_start, 0.5, -0.0025):
        try:
            z, _, odd = point_k(z, e, kappa_b_list[k])
        except Exception:                                 # end of the period-2 branch (fold at large kappa_b)
            break
        if odd < odd_min:                                 # merged with the period-1 satellite (period doubling)
            break
        g2_k.append(g2_of(e)/2/np.pi*1e3)
        mu_k.append(np.exp(floquet_exponents(z, e, kappa_b_list[k])*4*np.pi/w_a))   # T = 4 pi / w_a
    mu_plot[k] = (np.array(g2_k), np.array(mu_k))

fig, axes = plt.subplots(2, len(k_plot), figsize=(2*width_in, 1.15*width_in), layout="constrained", sharey="row")
norm = plt.Normalize(min(mu_plot[k][0].min() for k in k_plot), max(mu_plot[k][0].max() for k in k_plot))
th = np.linspace(0, 2*np.pi, 400)
for j, k in enumerate(k_plot):
    g2_k, mu_k = mu_plot[k]
    g2_star = g2_threshold_MHz[k]
    i_star = np.argmin(np.abs(g2_k - g2_star))
    # complex plane
    ax = axes[0, j]
    ax.plot(np.cos(th), np.sin(th), color="grey", lw=0.8)
    sc = ax.scatter(mu_k.real.ravel(), mu_k.imag.ravel(), c=np.repeat(g2_k, mu_k.shape[1]), cmap=Perso_colormap, norm=norm, s=4)
    ax.scatter(mu_k[i_star].real, mu_k[i_star].imag, s=25, facecolors="none", edgecolors="k", lw=0.8)
    ax.set_xlim(-0.3, 1.8); ax.set_ylim(-1.5, 1.5); ax.set_aspect("equal")
    ax.set_title(rf"$\kappa_b/2\pi={kappa_G_list[k]/2/np.pi*1e3:.0f}$ MHz", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.set_xlabel(r"$\mathrm{Re}\,\mu$", fontsize=7)
    if j == 0: ax.set_ylabel(r"$\mathrm{Im}\,\mu$", fontsize=7)
    # argument and modulus against g2 (upper half-plane members)
    ax = axes[1, j]
    up = mu_k.imag >= 0
    for i in range(mu_k.shape[1]):
        m = up[:, i]
        ax.scatter(g2_k[m], np.angle(mu_k[m, i])/np.pi, c=np.where(np.abs(mu_k[m, i]) > 1, "C4", "C2"), s=3)
    ax.tick_params(labelsize=6)
    ax.axvline(g2_star, ls="--", color="grey", lw=0.8)
    ax.set_xlabel(r"$g_2/2\pi$ (MHz)", fontsize=7)
    if j == 0: ax.set_ylabel(r"$\arg\mu/\pi$ ($\mathrm{Im}\,\mu\geq0$)", fontsize=7)
fig.colorbar(sc, ax=axes[0, :], label=r"$g_2/2\pi$ (MHz)")
axes[1, 0].scatter([], [], c="C4", s=6, label=r"$|\mu|>1$"); axes[1, 0].scatter([], [], c="C2", s=6, label=r"$|\mu|<1$")
axes[1, 0].legend(fontsize=6)
plt.show()

# multipliers along the sweep (upper half-plane). With two complex pairs, Liouville gives |mu_1||mu_2| = exp(-kappa_b T/2);
# the Hamiltonian-Hopf quartet also has arg mu_1 = arg mu_2
for k in k_plot:
    g2_k, mu_k = mu_plot[k]
    print(f"--- kappa_b/2pi = {kappa_G_list[k]/2/np.pi*1e3:.0f} MHz, threshold g2*/2pi = {g2_threshold_MHz[k]:.2f} MHz, "
          f"exp(-kappa_b T/2) = {np.exp(-kappa_G_list[k]*4*np.pi/w_a/2):.3f}")
    for i in list(range(0, len(g2_k), 8)) + [len(g2_k) - 1]:
        m = mu_k[i][mu_k[i].imag >= -1e-12]
        print(f"g2/2pi = {g2_k[i]:6.2f} MHz: |mu| = {np.round(np.abs(m), 3)}, arg/pi = {np.round(np.angle(m)/np.pi, 3)}")

**Floquet multipliers in the complex plane: Hamiltonian–Hopf unfolded by the loss, off the real axis.**
* $\kappa_b\to0$ (2 MHz): below the threshold the two pairs sit on the unit circle at different angles. They meet at $\arg\mu\simeq\pm0.20\pi$ (about $36^\circ$) for $g_2/2\pi\simeq76.2$ MHz,
  then leave the circle on the same ray, $|\mu|$ and $1/|\mu|$ (e.g. 1.289 and 0.773 at $\arg\mu=0.209\pi$, 77.6 MHz). This is the quartet $\mu,\mu^*,1/\mu^*,1/\mu$ of a Hamiltonian–Hopf (Krein) collision.
* With loss the pairs no longer collide. Both start inside the circle, their arguments stay apart (0.21π and 0.19π at 88 MHz; 0.23π and 0.07π at 500 MHz),
  and only one pair crosses $|\mu|=1$, at $\arg\mu\simeq\pm(0.21$–$0.24)\pi$: a Neimark–Sacker bifurcation, the dissipative unfolding of the Hamiltonian–Hopf.
  The moduli of the two pairs are tied by Liouville, $|\mu_1||\mu_2|=e^{-\kappa_bT/2}$, so the unstable pair needs $|\mu_1|>1$ against a mean modulus $e^{-\kappa_bT/4}<1$.
* The crossing is not on the real axis. A real crossing would be $\mu=-1$ (period doubling) or $\mu=+1$ (fold). On the real axis, for $\kappa_b/2\pi\gtrsim300$ MHz, there is only the *other* pair before the threshold:
  two real multipliers that collide on the positive real axis (81 MHz at 294 MHz, 95.3 MHz at 500 MHz) and become complex. This is a node-to-focus change of the stable directions, not a bifurcation.
  Near the low end of these branches, one real multiplier approaches $+1$ (0.99 at 294 MHz), so the period-2 branch ends at a fold there rather than at the period doubling.